In [1]:
!pip install -q transformers datasets accelerate huggingface_hub scikit-learn

## Dataset: CUAD (Contract Understanding Atticus Dataset)
13,000+ labeled clause spans across 41 categories from 510 real commercial contracts.
Loaded via HF's auto-converted parquet export (script-based loading was deprecated in
`datasets>=4.0`).

In [2]:
from datasets import load_dataset

dataset = load_dataset("theatticusproject/cuad-qa", revision="refs/convert/parquet")
print(dataset)

default/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 4.32MB            

default/train/0000.parquet: downloading bytes:           |  0.00B            

default/train/0001.parquet: reconstructing file:   0%|          |  0.00B / 4.08MB            

default/train/0001.parquet: downloading bytes:           |  0.00B            

default/train/0002.parquet: reconstructing file:   0%|          |  0.00B / 3.69MB            

...

default/test/0000.parquet: reconstructing file:   0%|          |  0.00B / 2.64MB            

default/test/0000.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 22450
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 4182
    })
})


In [3]:
import re
from collections import Counter

def extract_category(question):
    match = re.search(r'"([^"]+)"', question)
    return match.group(1) if match else None

pairs = []
for row in dataset["train"]:
    if len(row["answers"]["text"]) > 0:
        category = extract_category(row["question"])
        text = row["answers"]["text"][0]
        if category and text.strip():
            pairs.append({"text": text, "category": category})

print(f"Total labeled pairs: {len(pairs)}")
print(Counter(p["category"] for p in pairs).most_common(41))

Total labeled pairs: 11180
[('Parties', 2011), ('License Grant', 642), ('Cap On Liability', 554), ('Audit Rights', 538), ('Anti-Assignment', 517), ('Insurance', 443), ('Document Name', 419), ('Expiration Date', 384), ('Agreement Date', 383), ('Governing Law', 374), ('Post-Termination Services', 368), ('Effective Date', 363), ('Minimum Commitment', 336), ('Exclusivity', 332), ('Revenue/Profit Sharing', 331), ('Rofr/Rofo/Rofn', 299), ('Ip Ownership Assignment', 257), ('Non-Transferable License', 255), ('Termination For Convenience', 205), ('Non-Compete', 200), ('Change Of Control', 191), ('Renewal Term', 179), ('Warranty Duration', 157), ('Uncapped Liability', 151), ('Irrevocable Or Perpetual License', 142), ('Volume Restriction', 136), ('Covenant Not To Sue', 129), ('Notice Period To Terminate Renewal', 104), ('Joint Ip Ownership', 101), ('Competitive Restriction Exception', 98), ('Liquidated Damages', 98), ('Affiliate License-Licensee', 88), ('No-Solicit Of Employees', 73), ('Source Co

## Category Selection
Of 41 CUAD categories, selected the top 10 by frequency, then dropped "Parties" —
inspection showed 70% of its examples were short entity names (e.g. "Google",
"Distributor"), not clause-level legal text, making it inconsistent with the other
9 risk-relevant categories and a weaker signal for a clause classifier. Final set:
9 categories, ~250-650 examples each, all genuine clause-length text.

In [4]:
FINAL_CATEGORIES = [
    "License Grant", "Cap On Liability", "Audit Rights",
    "Anti-Assignment", "Insurance", "Governing Law",
    "Post-Termination Services", "Minimum Commitment", "Exclusivity"
]

filtered_pairs = [p for p in pairs if p["category"] in FINAL_CATEGORIES]
label2id = {cat: i for i, cat in enumerate(FINAL_CATEGORIES)}
id2label = {i: cat for i, cat in enumerate(FINAL_CATEGORIES)}
NUM_CLASSES = len(FINAL_CATEGORIES)

for p in filtered_pairs:
    p["label"] = label2id[p["category"]]

print(f"Filtered pairs: {len(filtered_pairs)}, NUM_CLASSES: {NUM_CLASSES}")
print(f"NUM_CLASSES = {NUM_CLASSES}")

Filtered pairs: 4104, NUM_CLASSES: 9
NUM_CLASSES = 9


In [5]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

texts = [p["text"] for p in filtered_pairs]
labels = [p["label"] for p in filtered_pairs]

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.15, random_state=42, stratify=labels
)

train_ds = Dataset.from_dict({"text": train_texts, "label": train_labels})
val_ds = Dataset.from_dict({"text": val_texts, "label": val_labels})
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

Train: 3488, Val: 616


## Why F1-macro, not accuracy
Class sizes range from ~250 to ~650 examples — a meaningful imbalance. Accuracy
would let the model coast on majority classes; F1-macro weights all 9 categories
equally, giving an honest measure of per-category performance.

In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)

train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok = val_ds.map(tokenize_fn, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=NUM_CLASSES, id2label=id2label, label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

args = TrainingArguments(
    output_dir="clause-classifier",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
    fp16=True,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_tok, eval_dataset=val_tok,
    compute_metrics=compute_metrics,
)

trainer.train()

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/3488 [00:00<?, ? examples/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.368603,0.294303,0.909091,0.904809
2,0.202139,0.272821,0.912338,0.909476
3,0.141328,0.245525,0.922078,0.920260


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=654, training_loss=0.3544112854047653, metrics={'train_runtime': 107.1327, 'train_samples_per_second': 97.673, 'train_steps_per_second': 6.105, 'total_flos': 693155949133824.0, 'train_loss': 0.3544112854047653, 'epoch': 3.0})

In [7]:
from huggingface_hub import notebook_login
notebook_login()

In [8]:
!pip install -q sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [9]:
model_name_on_hub = "Adi2335/nda-clause-classifier-v2"

trainer.model.push_to_hub(model_name_on_hub)
tokenizer.push_to_hub(model_name_on_hub)

print(f"Pushed to https://huggingface.co/{model_name_on_hub}")

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...l1ai9a9/model.safetensors:   2%|1         | 5.02MB /  268MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Pushed to https://huggingface.co/Adi2335/nda-clause-classifier-v2


In [10]:
import json
from google.colab import files

with open("nda_clauses_corpus.json", "w") as f:
    json.dump(filtered_pairs, f)

files.download("nda_clauses_corpus.json")
print(f"Exported {len(filtered_pairs)} records")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Exported 4104 records
